In [2]:
import pandas   as  pd
from openpyxl import load_workbook
import sys
import os

In [10]:
# Add the directory containing utils.py to the module search path
utils_path = os.path.abspath('../streamlit/src/')
sys.path.append(utils_path)

# Import the function from utils.py
from utils import analyze_merge

In [11]:
umsteiger_csv = pd.read_csv('/Users/leonardhaas/code/streamlit/data/raw_data/Umsteigeschluessel-KLDB2020-ISCO08.csv',sep=';',header=4)

In [12]:
umsteiger_csv['Bezeichnungen der ISCO-08 (4-Steller)'] = umsteiger_csv['ISCO-08\n(4-Steller)'].fillna('0').astype(int)

In [13]:
# TODO hwo much data do we lose when we ignore the undeutige umsteiger
umsteiger_csv['Umstieg eindeutig (1);\nnicht eindeutig (0)'].value_counts()

Umstieg eindeutig (1);\nnicht eindeutig (0)
1.0    1136
0.0     387
Name: count, dtype: int64

## eindeutig vs uneindeutig

In [14]:
umsteiger_csv.rename(
    columns={'Umstieg eindeutig (1);\nnicht eindeutig (0)': 'umstieg_eindeutig'},
    inplace=True
)

In [15]:
eindeutig = umsteiger_csv.query("umstieg_eindeutig==1")

id_realtion=eindeutig[['KldB 2010\n(5-Steller)','ISCO-08\n(4-Steller)']][:1521]
id_realtion.tail()
id_realtion.columns = ['kldb_2010_key','isco_08_key']

In [16]:
id_realtion['kldb_2010_key'] =id_realtion['kldb_2010_key'].fillna(0).astype(int)

In [17]:
column_dtypes = {

    'bezeichnung_kldb': 'string',    # For text data
    'median_brutto': 'int64',      # For numeric data (int64)
    'average_brutto': 'int64',     # For numeric data (int64)
    'code_kldb': 'int64'             # For integer data
}

### old verdienst 2023

In [10]:
verdienst_data_destatis= pd.read_csv('/Users/leonardhaas/code/streamlit/data/raw_data/verdienst_destatis_kldb_5steller.csv')#,dtype=column_dtypes)
verdienst_data_destatis.drop(columns=['Unnamed: 0'], inplace=True)


In [11]:
for column, dtype in column_dtypes.items():
    if dtype == 'int64':
        # Fill NaN with a default value (e.g., 0) before converting
        verdienst_data_destatis[column] = verdienst_data_destatis[column].fillna(0).astype(dtype)
    else:
        # For other types like string, direct conversion works
        verdienst_data_destatis[column] = verdienst_data_destatis[column].astype(dtype)

In [12]:
verdienst_data_destatis = verdienst_data_destatis[(verdienst_data_destatis['median_brutto'] != 0) | (verdienst_data_destatis['average_brutto'] != 0)]

In [13]:
id_verdienst_data_destatis,merge_report=analyze_merge(verdienst_data_destatis,id_realtion,'code_kldb','kldb_2010_key')

Merge Analysis Report:
Total rows in left dataset: 1105
Unique keys in left dataset: 1105
Matched unique keys: 957
Unmatched unique keys: 148
Total rows in merged dataset: 1105
Match Percentage (unique keys): 86.61%

Merge Counts:
both: 957
left_only: 148
right_only: 0


### new verdienst 2024

In [4]:
verdienst_data_2024 = pd.read_csv("../data/raw_data/verdienste_2024_KdB_5.csv",sep=';')

In [20]:
verdienst_data_2024['code_kldb']=verdienst_data_2024['2_Auspraegung_Code'].str.replace('KB10-','')
verdienst_data_2024['code_kldb'] = verdienst_data_2024['code_kldb'].astype(float)

In [ ]:
# Show how the merge was
verdienst_data_isco_codes,merge_report=analyze_merge(verdienst_data_2024,id_realtion,'code_kldb','kldb_2010_key')

Merge Analysis Report:
Total rows in left dataset: 1301
Unique keys in left dataset: 1300
Matched unique keys: 1136
Unmatched unique keys: 164
Total rows in merged dataset: 1301
Match Percentage (unique keys): 87.38%

Merge Counts:
both: 1136
left_only: 165
right_only: 0


,Statistik_Code,Statistik_Label,Zeit_Code,Zeit_Label,Zeit,1_Merkmal_Code,1_Merkmal_Label,1_Auspraegung_Code,1_Auspraegung_Label,2_Merkmal_Code,2_Merkmal_Label,2_Auspraegung_Code,2_Auspraegung_Label,VST047__Durchschn_Bruttomonatsverdienste_ohne_Sonderz__EUR,VST052__Mittlere_Bruttomonatsverdienste_ohne_Sonderz__EUR,code_kldb,kldb_2010_key,isco_08_key
0,62361,Verdiensterhebung,SMONAT,Stichmonat,Apr 23,DINSG,Deutschland insgesamt,DG,Deutschland,KB10A5,"Berufsgattungen (KB2010), 5-Steller",KB10-11101,Landwirtschaft (oS) - Helfer,2282,2222,11101.0,NaN,NaN
1,62361,Verdiensterhebung,SMONAT,Stichmonat,Apr 23,DINSG,Deutschland insgesamt,DG,Deutschland,KB10A5,"Berufsgattungen (KB2010), 5-Steller",KB10-11102,Landwirtschaft (oS) - Fachkraft,2609,2550,11102.0,NaN,NaN
2,62361,Verdiensterhebung,SMONAT,Stichmonat,Apr 23,DINSG,Deutschland insgesamt,DG,Deutschland,KB10A5,"Berufsgattungen (KB2010), 5-Steller",KB10-11103,Landwirtschaft (oS) - Spezialist,/,/,11103.0,11103.0,2132.0
3,62361,Verdiensterhebung,SMONAT,Stichmonat,Apr 23,DINSG,Deutschland insgesamt,DG,Deutschland,KB10A5,"Berufsgattungen (KB2010), 5-Steller",KB10-11104,Landwirtschaft (oS) - Experte,5791,5331,11104.0,11104.0,2132.0
4,62361,Verdiensterhebung,SMONAT,Stichmonat,Apr 23,DINSG,Deutschland insgesamt,DG,Deutschland,KB10A5,"Berufsgattungen (KB2010), 5-Steller",KB10-11113,Landtechnik - Spezialist,3848,3447,11113.0,11113.0,3142.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1296,62361,Verdiensterhebung,SMONAT,Stichmonat,Apr 23,DINSG,Deutschland insgesamt,DG,Deutschland,KB10A5,"Berufsgattungen (KB2010), 5-Steller",KB10-01104,Offiziere,5564,5330,1104.0,1104.0,110.0
1297,62361,Verdiensterhebung,SMONAT,Stichmonat,Apr 23,DINSG,Deutschland insgesamt,DG,Deutschland,KB10A5,"Berufsgattungen (KB2010), 5-Steller",KB10-01203,Unteroffiziere mit Portepee,3815,3785,1203.0,1203.0,210.0
1298,62361,Verdiensterhebung,SMONAT,Stichmonat,Apr 23,DINSG,Deutschland insgesamt,DG,Deutschland,KB10A5,"Berufsgattungen (KB2010), 5-Steller",KB10-01302,Unteroffiziere ohne Portepee,3089,3029,1302.0,1302.0,210.0
1299,62361,Verdiensterhebung,SMONAT,Stichmonat,Apr 23,DINSG,Deutschland insgesamt,DG,Deutschland,KB10A5,"Berufsgattungen (KB2010), 5-Steller",KB10-01402,Angehörige reguläre Streitkräfte in sonst. Rängen,2704,2627,1402.0,1402.0,310.0


## problem kind of duplicates when calculating average

In [18]:
fraktion_verdienst_data['median_brutto_group_mean'] = fraktion_verdienst_data.groupby('isco_08_key')['median_brutto'].transform('mean')

result = fraktion_verdienst_data.groupby([
    'isco_08_key',
    'fraktion',
    'Berufsgattung(ISCO-Stufe 4)',
    'Anzahl'
]).agg({
    'median_brutto_group_mean': 'first'
}).reset_index()


In [50]:
result.to_csv('../streamlit/data/processed_data/isco_verdienst.csv')